In [1]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.TreeTimeML import TreeTimeML
from src.predictionModule.LoadupSamples import LoadupSamples
from src.hyperparameterTuning.HelperMetrics import HelperMetrics
from src.hyperparameterTuning.BaseStrategy import BaseStrategy
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels

import pandas as pd
import numpy as np
import polars as pl
import datetime
import seaborn as sns
import lightgbm as lgb
import random
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import r_regression

import logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(message)s'
)
logger = logging.getLogger(__name__)

c:\Users\kimer\Desktop\RandomOdyssey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.common.AssetData import AssetData
from src.common.AssetDataService import AssetDataService
from src.common.AssetFileInOut import AssetFileInOut 
from src.common.AssetDataPolars import AssetDataPolars 
from src.databaseService.EstablishStocks import EstablishStocks 
from src.databaseService.OutsourceLoader import OutsourceLoader
from src.databaseService.CleanData import CleanData 

assets=AssetFileInOut("../src/stockGroups/bin").loadDictFromFile("group_dez_lowspread")

# Convert to Polars for speedup
assetspl: dict[str, AssetDataPolars] = {}
for ticker, asset in assets.items():
    assetspl[ticker]= AssetDataService.to_polars(asset)
    
tickers = list(assetspl.keys())

In [15]:
test_asset = assetspl['TXN']
test_sp = test_asset.shareprice
global_start_date = datetime.date(2023, 1, 1)
final_eval_date = datetime.date(2025, 11, 24)
n_test_days=7
test_dates = [final_eval_date - datetime.timedelta(days=i) for i in range(n_test_days)][::-1]
train_end_date = min(test_dates) - datetime.timedelta(days=1)

test_sp = test_sp.filter((pl.col("Date") >= global_start_date) & (pl.col("Date") <= train_end_date))
test_sp.columns

['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'AdjClose',
 'Volume',
 'Dividends',
 'Splits']

In [16]:
load_start_date = global_start_date - datetime.timedelta(days=180)
load_end_date = train_end_date + datetime.timedelta(days=20)
shareprices = {t: (
        assetspl[t].shareprice.filter(
            (pl.col("Date") >= load_start_date) & (pl.col("Date") <= load_end_date)
        )
    ) 
    for t in tickers
}
def to_long_close(shareprices: dict[str, pl.DataFrame]) -> pl.DataFrame:
    frames = []
    for ticker, df in shareprices.items():
        frames.append(
            df.select([
                pl.col("Date").alias("date"),
                pl.lit(ticker).alias("ticker"),
                pl.col("Close").alias("Close"),
                pl.col("Open").alias("Open"),
                pl.col("High").alias("High"),
                pl.col("Low").alias("Low"),
                pl.col("AdjClose").alias("adjClose"),
            ])
        )

    return (
        pl.concat(frames, how="vertical")
        .sort(["date", "ticker"])
    )

meta_atr = to_long_close(shareprices)
meta_atr = meta_atr.with_columns(
    (pl.col("adjClose") / pl.col("Close")).alias("adjFactor")
).with_columns(
    (pl.col("Open") * pl.col("adjFactor")).alias("adjOpen"),
    (pl.col("High") * pl.col("adjFactor")).alias("adjHigh"),
    (pl.col("Low")  * pl.col("adjFactor")).alias("adjLow"),
).drop("adjFactor")

# optional: reorder columns
meta_atr = meta_atr.select([
    "date", "ticker",
    "Open", "High", "Low", "Close",
    "adjOpen", "adjHigh", "adjLow", "adjClose",
])

In [17]:
def add_wilder_atr(meta_atr: pl.DataFrame, window: int = 14) -> pl.DataFrame:
    out = (
        meta_atr
        .with_columns(
            pl.col("adjClose")
              .shift(1)
              .over("ticker", order_by="date")
              .alias("prev_adjClose")
        )
        .with_columns(
            # Avoid null issues on the first row per ticker
            pl.col("prev_adjClose")
              .fill_null(pl.col("adjClose"))
              .alias("_prev_used")
        )
        .with_columns(
            pl.max_horizontal([
                (pl.col("adjHigh") - pl.col("adjLow")),
                (pl.col("adjHigh") - pl.col("_prev_used")).abs(),
                (pl.col("adjLow")  - pl.col("_prev_used")).abs(),
            ]).alias("tr")
        )
        .with_columns(
            pl.cum_count("tr").over("ticker", order_by="date").alias("_idx"),
            pl.col("tr").rolling_mean(window).over("ticker", order_by="date").alias("_tr_sma"),
        )
        .with_columns(
            # Seed Wilder ATR with SMA at index window-1, then feed TR thereafter
            pl.when(pl.col("_idx") < window - 1).then(None)
              .when(pl.col("_idx") == window - 1).then(pl.col("_tr_sma"))
              .otherwise(pl.col("tr"))
              .alias("_tr_seeded")
        )
        .with_columns(
            pl.col("_tr_seeded")
              .ewm_mean(alpha=1.0 / window, adjust=False, ignore_nulls=True)
              .over("ticker", order_by="date")
              .alias("atr")
        )
        .drop(["_prev_used", "_idx", "_tr_sma", "_tr_seeded"])
    )

    return out

atr_n = 10
meta_atr = add_wilder_atr(meta_atr, window=atr_n)

# optional: reorder
meta_atr = meta_atr.select([
    "date", "ticker",
    "Open", "High", "Low", "Close",
    "adjOpen", "adjHigh", "adjLow", "adjClose",
    "tr", "atr"
])

In [21]:
n_timesteps = 90
meta_ext = (
    meta_atr
    .with_columns([
        (pl.col("adjClose").shift(m)/pl.col("adjClose")).over("ticker", order_by="date").alias(f"adjClose_tree_m{m}") 
        for m in range(0, n_timesteps)
    ]).filter(
        (pl.col("date") >= global_start_date) & (pl.col("date") <= train_end_date)
    )
)
Xtree_close = meta_ext.select([f"adjClose_tree_m{m}" for m in range(0, n_timesteps)][::-1]).to_numpy()
feature_atr = meta_ext.get_column("atr").to_numpy()
dates = meta_ext.get_column("date")

n_tar = 8
meta_sol = (
    meta_atr.with_columns(
        (pl.col("adjOpen").shift(-1)).over("ticker", order_by="date").alias(f"nextDayAdjOpen") 
    )
    .with_columns([
        (pl.col("adjClose").shift(-p)/pl.col("nextDayAdjOpen")).over("ticker", order_by="date").alias(f"adjClose_tar_p{p}") 
        for p in range(1, n_tar+1)
    ]).filter(
        (pl.col("date") >= global_start_date) & (pl.col("date") <= train_end_date)
    )
)

ytree = meta_sol.select([f"adjClose_tar_p{p}" for p in range(1, n_tar+1)]).to_numpy()
z = None
Z = None

In [20]:
eval = HelperMetrics.evaluate_mask_nullonempty(np.ones(Xtree_close.shape[0], dtype=bool), dates, ytree[:, 4])

print(f"Initial eval: {eval}")
print(f"Shape Xtree_close: {Xtree_close.shape}")
print(f"Shape ytree: {ytree.shape}")
print(f"Number of tickers {meta_ext.get_column("ticker").n_unique()}")
print(f"Number of dates {meta_ext.get_column("date").n_unique()}")

if not meta_atr.get_column("date").is_sorted():
    print("meta_atr not sorted!")
    
if not dates.is_sorted():
    print("Dates not sorted!")

Initial eval: 1.0022973462667126
Shape Xtree_close: (70034, 90)
Shape ytree: (70034, 8)
Number of tickers 97
Number of dates 722


In [14]:
import optuna
def moving_average(prices, window):
    nS, nT = prices.shape
    ma = np.empty_like(prices, dtype=float)

    cumsum = np.cumsum(prices, axis=1, dtype=float)
    for t in range(nT):
        start = max(0, t - window + 1)
        count = t - start + 1

        total = cumsum[:, t] - (cumsum[:, start - 1] if start > 0 else 0.0)
        ma[:, t] = total / count
    return ma
def exponential_moving_average(prices, span):
    nS, nT = prices.shape
    ema = np.empty_like(prices, dtype=float)

    alpha = 2.0 / (span + 1.0)
    ema[:, 0] = prices[:, 0]

    for t in range(1, nT):
        ema[:, t] = alpha * prices[:, t] + (1.0 - alpha) * ema[:, t - 1]
    return ema
def time_error(X_close, X_ma, wndw=10):
    nS, nT = X_close.shape
    
    rmse = np.empty(nS, dtype=float)
    
    rmse = np.sqrt(np.mean((X_close[:, -wndw:] - X_ma[:, -wndw:]) ** 2, axis=1))
    
    return rmse
def objective(trial):
    ma_wndw    = trial.suggest_int("ma_wndw", 1, 30, step=1) 
    slope_wndw   = trial.suggest_int("end_wndw", 1, 31, step=1)

    qup_atr    = trial.suggest_float("qup_atr", 0.86, 0.94)
    qdown_atr  = 0.9985 #trial.suggest_float("qdown_atr", 0.99, 0.999)
    
    qup_slope   = trial.suggest_float("qup_slope", 0.55, 0.72)
    qdown_slope = 0.9985 #trial.suggest_float("qdown_slope", 0.99, 0.999)
    
    qup_rmse    = trial.suggest_float("qup_rmse", 0.5, 0.8)
    qdown_rmse  = 0.998 #trial.suggest_float("qdown_rmse", 0.98, 0.999)
    
    #qup_sit = trial.suggest_float("qup_sit", 0.0, 0.92)
    #qdown_sit = trial.suggest_float("qdown_sit", qup_sit+0.05, 0.999)
    #qup_sit = np.clip(qup_sit, 0.0, 0.999)
    #qdown_sit = np.clip(qdown_sit, 0.0, 0.999)
    
    rsme_wndw  = trial.suggest_int("rsme_wndw", 4, 37, step=1)
    tunnel_delay = trial.suggest_int("tunnel_delay", 1, 6, step=1)
    
    atr_n = trial.suggest_int("atr_n", 10, 50, step=1)
    
    is_expo = False # trial.suggest_categorical("is_expo", [True, False])

    # --- compute MA, RMSE, slope ---
    if is_expo:
        X_ma = exponential_moving_average(Xtree_close, ma_wndw)
    else:
        X_ma = moving_average(Xtree_close, ma_wndw)

    # --- RMSE mask ---
    rmse = time_error(Xtree_close[:, :-tunnel_delay], X_ma[:, :-tunnel_delay], wndw=rsme_wndw)               # shape (nS,)
    thr_rmse_up, thr_rmse_down = np.quantile(rmse, [qup_rmse, qdown_rmse])
    mask_rmse = (rmse >= thr_rmse_up) & (rmse <= thr_rmse_down)
    
    # --- slope mask ---
    slope = np.mean(np.diff(Xtree_close[:, -slope_wndw:], axis=1),axis=1)  # shape (nS,)
    thr_slope_up, thr_slope_down = np.quantile(slope, [qup_slope, qdown_slope])
    mask_slope = (slope >= thr_slope_up) & (slope <= thr_slope_down)

    # --- atr mask ---
    meta_atr_tmp = add_wilder_atr(meta_atr, window=atr_n).filter(
        (pl.col("date") >= global_start_date) & (pl.col("date") <= train_end_date)
    )
    atr = meta_atr_tmp.get_column("atr").to_numpy()
    thr_atr_up, thr_atr_down = np.quantile(atr, [qup_atr, qdown_atr])
    mask_atr = (atr >= thr_atr_up) & (atr <= thr_atr_down)
    
    # --- current situation ---
    #sit = X_ma[:, -1] - Xtree_close[:, -1]
    #thr_sit_up, thr_sit_down = np.quantile(sit, [qup_sit, qdown_sit])
    #mask_sit = (sit >= thr_sit_up) & (sit <= thr_sit_down)
    

    # --- combined mask ---
    mask_atrtunnel = mask_rmse & mask_slope & mask_atr #& mask_sit

    if not dates.is_sorted():
        raise optuna.exceptions.TrialPruned()
    metric = HelperMetrics.evaluate_mask_oneonempty(
        mask_atrtunnel,
        dates,
        ytree[:, 4],
    )
    rolling_logmean = (
        pl.DataFrame({
            "date": dates,
            "y": ytree[:, 4]
        }).filter(pl.Series(mask_atrtunnel))
        .group_by("date")
        .agg(pl.col("y").mean().alias("y_mean"))
        .select(pl.col("y_mean").log().rolling_mean(window_size=100).drop_nulls().alias("y_rolllogmean"))
    )
    rolling_logmean_np = rolling_logmean.to_numpy()
    rolling_logmean_ge0 = (rolling_logmean
        .select(pl.col("y_rolllogmean").ge(pl.lit(0.0)).alias("y_rolllogmean_ge0"))
        .to_numpy()
    )
    
    std = np.std(rolling_logmean_np - np.mean(rolling_logmean_np))
    ge0_ratio = rolling_logmean_ge0.sum() / len(rolling_logmean_ge0)
    score = metric

    trial.set_user_attr("n_selected", int(mask_atrtunnel.sum()))
    trial.set_user_attr("n_ratio", int(mask_atrtunnel.sum()) / Xtree_close.shape[0])
    trial.set_user_attr("metric", metric)
    trial.set_user_attr("std_rolllogmean_ge0", std)
    trial.set_user_attr("ge0_ratio", ge0_ratio)
    trial.set_user_attr("score", score)

    return score

sampler = optuna.samplers.TPESampler(n_startup_trials=50)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=500)
df_atrtunnel: pd.DataFrame = study.trials_dataframe()
print("Best value:", study.best_value)
print("Best params:", study.best_params)

[I 2025-12-09 16:23:20,991] A new study created in memory with name: no-name-bd488b24-1aa3-4446-ac24-c7f31b89bb1f
[I 2025-12-09 16:23:21,066] Trial 0 finished with value: 0.9987027779817974 and parameters: {'ma_wndw': 3, 'end_wndw': 15, 'qup_atr': 0.8936944225259222, 'qup_slope': 0.6879004630939871, 'qup_rmse': 0.6999068874037941, 'rsme_wndw': 10, 'tunnel_delay': 2, 'atr_n': 30}. Best is trial 0 with value: 0.9987027779817974.
[I 2025-12-09 16:23:21,131] Trial 1 finished with value: 0.9977608608132176 and parameters: {'ma_wndw': 22, 'end_wndw': 26, 'qup_atr': 0.9385433965296932, 'qup_slope': 0.6995195897968304, 'qup_rmse': 0.6140161069801774, 'rsme_wndw': 29, 'tunnel_delay': 2, 'atr_n': 35}. Best is trial 0 with value: 0.9987027779817974.
[I 2025-12-09 16:23:21,192] Trial 2 finished with value: 1.0036219472608046 and parameters: {'ma_wndw': 1, 'end_wndw': 12, 'qup_atr': 0.8807274159508233, 'qup_slope': 0.586555271424428, 'qup_rmse': 0.5906684558578187, 'rsme_wndw': 30, 'tunnel_delay': 

Best value: 1.0054570476309288
Best params: {'ma_wndw': 18, 'end_wndw': 11, 'qup_atr': 0.8600883201471528, 'qup_slope': 0.703804072211104, 'qup_rmse': 0.6352746911301466, 'rsme_wndw': 25, 'tunnel_delay': 4, 'atr_n': 10}
